In [10]:
import os
import json
import torch
import pickle
import numpy as np
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, random_split
from typing import Dict, List, Tuple, Optional, Set
from base import BaseDataLoader, DatasetSubset


In [13]:
class RedditDataset(Dataset):
    def __init__(
        self, 
        data_dir: str, 
        vocab_path: str,
        split: str = 'train',
        seq_length: int = 10,  # Default to 10 as per data format
        iid: bool = True,
        num_clients: Optional[int] = None
    ):
        super().__init__()
        self.seq_length = seq_length
        self.split = split
        self.iid = iid
        self.train = split == 'train'
        
        # Load vocabulary
        with open(vocab_path, 'rb') as f:
            vocab_dict = pickle.load(f)
            self.vocab = vocab_dict['vocab']
            self.pad_idx = vocab_dict['pad_symbol']
            self.unk_idx = vocab_dict['unk_symbol']
            self.vocab_size = len(self.vocab)

        # Load data and create client splits if needed
        self.sequences, self.client_indices = self._load_and_prepare_data(
            data_dir, 
            num_clients if self.train else None
        )

    def _load_and_prepare_data(
        self, 
        data_dir: str,
        num_clients: Optional[int] = None
    ) -> Tuple[List[torch.Tensor], Optional[Dict[int, Set[int]]]]:
        """Load and prepare data from JSON files"""
        sequences = []
        json_files = [f for f in os.listdir(data_dir) 
                     if f.endswith(f'_{self.split}.json')]
        json_files.sort()

        # Process each file
        for json_file in json_files:
            with open(os.path.join(data_dir, json_file)) as f:
                data = json.load(f)
                user_data = data['user_data']

                # Process each user's sequences
                for user_id, user_content in user_data.items():
                    for sequence_list in user_content['x']:
                        # Handle nested list structure
                        if isinstance(sequence_list[0], list):
                            sequence = sequence_list[0]  # Take first sequence if nested
                        else:
                            sequence = sequence_list

                        # Convert tokens to indices
                        token_indices = []
                        for token in sequence:
                            if isinstance(token, str):
                                token_indices.append(self.vocab.get(token, self.unk_idx))
                            else:
                                token_indices.append(self.unk_idx)

                        # Ensure sequence length is correct
                        if len(token_indices) > self.seq_length:
                            token_indices = token_indices[:self.seq_length]
                        elif len(token_indices) < self.seq_length:
                            token_indices.extend([self.pad_idx] * (self.seq_length - len(token_indices)))

                        sequences.append(torch.tensor(token_indices))

        # Create client splits for training data
        client_indices = None
        if self.train and num_clients is not None:
            if self.iid:
                # IID split: randomly distribute sequences among clients
                indices = list(range(len(sequences)))
                np.random.shuffle(indices)
                
                samples_per_client = len(indices) // num_clients
                remaining_samples = len(indices) % num_clients
                
                client_indices = {}
                current_idx = 0
                
                for i in range(num_clients):
                    extra_sample = 1 if i < remaining_samples else 0
                    client_size = samples_per_client + extra_sample
                    
                    end_idx = current_idx + client_size
                    client_indices[i] = set(indices[current_idx:end_idx])
                    current_idx = end_idx
            else:
                # Non-IID split: maintain user-based grouping
                raise NotImplementedError("Non-IID splitting not implemented yet")

        return sequences, client_indices

    def __len__(self) -> int:
        return len(self.sequences)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Get input sequence and target for language modeling"""
        sequence = self.sequences[idx]
        
        # Input is all tokens except last
        x = sequence[:-1]
        # Target is all tokens except first
        y = sequence[1:]
        
        return x, y

    def get_vocab_size(self) -> int:
        return self.vocab_size

    def decode_sequence(self, indices: torch.Tensor) -> List[str]:
        """Convert a sequence of indices back to tokens"""
        idx_to_token = {idx: token for token, idx in self.vocab.items()}
        return [idx_to_token.get(idx.item(), '<UNK>') for idx in indices]

    def create_padding_mask(self, sequence: torch.Tensor) -> torch.Tensor:
        """Create padding mask for transformer attention"""
        return (sequence == self.pad_idx)

In [14]:

class RedditDataLoader(BaseDataLoader):
    def __init__(
        self, 
        data_dir: str,
        n_clients: int,
        batch_size: int,
        seq_length: int = 10,
        iid: bool = True
    ):
        """
        Reddit data loader inheriting from BaseDataLoader.
        
        Args:
            data_dir: Root directory containing train/val/test subdirectories
            vocab_path: Path to vocabulary file
            n_clients: Number of federated clients
            batch_size: Batch size for dataloaders
            seq_length: Sequence length (default 10 based on data format)
            iid: Whether to use IID data distribution
        """
        super().__init__(data_dir, n_clients, batch_size, iid)
        self.vocab_path = data_dir + '/reddit_vocab.pck'
        self.seq_length = seq_length

    def load_data(self) -> Tuple[List[DataLoader], DataLoader, DataLoader, int]:
        """
        Load and prepare the dataset.
        
        Returns:
            - List of training DataLoaders (one per client)
            - Validation DataLoader
            - Test DataLoader
            - Number of input channels/features (vocab size in this case)
        """
        # Load datasets
        train_dataset = RedditDataset(
            data_dir=os.path.join(self.data_dir, 'reddit_leaf/train'),
            vocab_path=self.vocab_path,
            split='train',
            seq_length=self.seq_length,
            iid=self.iid,
            num_clients=self.n_clients
        )

        val_dataset = RedditDataset(
            data_dir=os.path.join(self.data_dir, 'reddit_leaf/val'),
            vocab_path=self.vocab_path,
            split='val',
            seq_length=self.seq_length
        )

        test_dataset = RedditDataset(
            data_dir=os.path.join(self.data_dir, 'reddit_leaf/test'),
            vocab_path=self.vocab_path,
            split='test',
            seq_length=self.seq_length
        )

        # Create client-specific training dataloaders
        train_loaders = [
            DataLoader(
                DatasetSubset(train_dataset, train_dataset.client_indices[i]),
                batch_size=self.batch_size,
                shuffle=True
            ) for i in range(self.n_clients)
        ]

        # Create validation and test loaders
        val_loader = DataLoader(
            val_dataset,
            batch_size=self.batch_size,
            shuffle=False
        )
        
        test_loader = DataLoader(
            test_dataset,
            batch_size=self.batch_size,
            shuffle=False
        )

        return train_loaders, val_loader, test_loader, train_dataset.get_vocab_size()

In [15]:
import os
import torch
import numpy as np
from collections import Counter
from typing import List, Dict, Tuple



In [18]:
# Cell 1: Initialize the analyzer with basic setup
import torch
import numpy as np
from collections import Counter
from typing import List, Dict, Tuple

# Initialize the analyzer
data_dir = '../data/Reddit'
vocab_path = '../data/Reddit/reddit_vocab.pck'
n_clients = 10

data_loader = RedditDataLoader(
    data_dir=data_dir,
    n_clients=n_clients,
    batch_size=32,
    seq_length=10
)

# Get dataloaders
train_loaders, val_loader, test_loader, vocab_size = data_loader.load_data()

# Get pad_idx from first client's dataset (accessing through SubsetDataset wrapper)
pad_idx = train_loaders[0].dataset.dataset.pad_idx